In [2]:
import pandas as pd

# Column names for NSL-KDD (41 features + label + difficulty)
col_names = [
    "duration","protocol_type","service","flag","src_bytes","dst_bytes",
    "land","wrong_fragment","urgent","hot","num_failed_logins","logged_in",
    "num_compromised","root_shell","su_attempted","num_root","num_file_creations",
    "num_shells","num_access_files","num_outbound_cmds","is_host_login",
    "is_guest_login","count","srv_count","serror_rate","srv_serror_rate",
    "rerror_rate","srv_rerror_rate","same_srv_rate","diff_srv_rate",
    "srv_diff_host_rate","dst_host_count","dst_host_srv_count",
    "dst_host_same_srv_rate","dst_host_diff_srv_rate","dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate","dst_host_serror_rate","dst_host_srv_serror_rate",
    "dst_host_rerror_rate","dst_host_srv_rerror_rate","label","difficulty"
]

train_df = pd.read_csv("KDDTrain+.txt", names=col_names)
test_df = pd.read_csv("KDDTest+.txt", names=col_names)

print(train_df.shape)
print(test_df.shape)
print(train_df.head())

(125973, 43)
(22544, 43)
   duration protocol_type   service flag  src_bytes  dst_bytes  land  \
0         0           tcp  ftp_data   SF        491          0     0   
1         0           udp     other   SF        146          0     0   
2         0           tcp   private   S0          0          0     0   
3         0           tcp      http   SF        232       8153     0   
4         0           tcp      http   SF        199        420     0   

   wrong_fragment  urgent  hot  ...  dst_host_same_srv_rate  \
0               0       0    0  ...                    0.17   
1               0       0    0  ...                    0.00   
2               0       0    0  ...                    0.10   
3               0       0    0  ...                    1.00   
4               0       0    0  ...                    1.00   

   dst_host_diff_srv_rate  dst_host_same_src_port_rate  \
0                    0.03                         0.17   
1                    0.60                      

In [3]:
# Check what labels exist
print(train_df['label'].value_counts())

# Convert to binary classification: normal vs attack
train_df['binary_label'] = train_df['label'].apply(lambda x: 'normal' if x == 'normal' else 'attack')
test_df['binary_label'] = test_df['label'].apply(lambda x: 'normal' if x == 'normal' else 'attack')

print(train_df['binary_label'].value_counts())

label
normal             67343
neptune            41214
satan               3633
ipsweep             3599
portsweep           2931
smurf               2646
nmap                1493
back                 956
teardrop             892
warezclient          890
pod                  201
guess_passwd          53
buffer_overflow       30
warezmaster           20
land                  18
imap                  11
rootkit               10
loadmodule             9
ftp_write              8
multihop               7
phf                    4
perl                   3
spy                    2
Name: count, dtype: int64
binary_label
normal    67343
attack    58630
Name: count, dtype: int64


In [4]:
from sklearn.preprocessing import LabelEncoder

# Encode categorical columns
categorical_cols = ['protocol_type', 'service', 'flag']
for col in categorical_cols:
    le = LabelEncoder()
    train_df[col] = le.fit_transform(train_df[col])
    # Handle unseen categories in test set
    test_df[col] = test_df[col].map(lambda s: le.transform([s])[0] if s in le.classes_ else -1)

print(train_df[categorical_cols].head())

   protocol_type  service  flag
0              1       20     9
1              2       44     9
2              1       49     5
3              1       24     9
4              1       24     9


In [5]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Define features (X) and target (y)
feature_cols = [c for c in train_df.columns if c not in ['label', 'binary_label', 'difficulty']]

X = train_df[feature_cols]
y = train_df['binary_label']

X_test_final = test_df[feature_cols]
y_test_final = test_df['binary_label']

# Scale numeric features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test_final)

# Split training data into train/validation (80/20)
X_train, X_val, y_train, y_val = train_test_split(
    X_scaled, y, test_size=0.2, stratify=y, random_state=42
)

print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Final test shape:", X_test_scaled.shape)

Train shape: (100778, 41)
Validation shape: (25195, 41)
Final test shape: (22544, 41)


In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_val)

print(classification_report(y_val, y_pred))
print(confusion_matrix(y_val, y_pred))

              precision    recall  f1-score   support

      attack       1.00      1.00      1.00     11726
      normal       1.00      1.00      1.00     13469

    accuracy                           1.00     25195
   macro avg       1.00      1.00      1.00     25195
weighted avg       1.00      1.00      1.00     25195

[[11712    14]
 [    6 13463]]


In [7]:
y_test_pred = clf.predict(X_test_scaled)

print(classification_report(y_test_final, y_test_pred))
print(confusion_matrix(y_test_final, y_test_pred))

              precision    recall  f1-score   support

      attack       0.97      0.64      0.77     12833
      normal       0.67      0.97      0.79      9711

    accuracy                           0.78     22544
   macro avg       0.82      0.80      0.78     22544
weighted avg       0.84      0.78      0.78     22544

[[8150 4683]
 [ 280 9431]]


In [8]:
import pandas as pd

importances = pd.Series(clf.feature_importances_, index=feature_cols)
importances_sorted = importances.sort_values(ascending=False)
print(importances_sorted.head(15))

src_bytes                      0.197804
dst_bytes                      0.102625
same_srv_rate                  0.077252
dst_host_srv_count             0.075251
dst_host_same_srv_rate         0.068613
flag                           0.068557
logged_in                      0.042391
serror_rate                    0.037283
srv_serror_rate                0.034886
protocol_type                  0.032410
diff_srv_rate                  0.029932
dst_host_same_src_port_rate    0.029459
dst_host_diff_srv_rate         0.026653
service                        0.024937
count                          0.024236
dtype: float64


In [9]:
from sklearn.ensemble import GradientBoostingClassifier

gb_clf = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb_clf.fit(X_train, y_train)

y_test_pred_gb = gb_clf.predict(X_test_scaled)
print(classification_report(y_test_final, y_test_pred_gb))

              precision    recall  f1-score   support

      attack       0.97      0.65      0.78     12833
      normal       0.68      0.97      0.80      9711

    accuracy                           0.79     22544
   macro avg       0.82      0.81      0.79     22544
weighted avg       0.84      0.79      0.78     22544



In [10]:
clf_balanced = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)
clf_balanced.fit(X_train, y_train)

y_test_pred_bal = clf_balanced.predict(X_test_scaled)
print(classification_report(y_test_final, y_test_pred_bal))

              precision    recall  f1-score   support

      attack       0.97      0.62      0.75     12833
      normal       0.66      0.97      0.79      9711

    accuracy                           0.77     22544
   macro avg       0.81      0.80      0.77     22544
weighted avg       0.83      0.77      0.77     22544



In [11]:
import joblib
import os

os.makedirs("models", exist_ok=True)
joblib.dump(clf, "models/nids_rf.pkl")
joblib.dump(scaler, "models/scaler.pkl")

print("Model and scaler saved!")

Model and scaler saved!
